## 12.3 三次元空間のグラフ

In [1]:
import json
import numpy as np
np.random.seed(0)
from scipy.integrate import solve_ivp

from plotly import graph_objects as go
from plotly.graph_objs.layout import Template

def func_lorentz(t:np.ndarray, q:np.ndarray)->np.ndarray:
    """Lorentz関数

    Args:
        t (np.ndarray): 時刻（使用しないがSciPyのソルバーのインターフェースに合わせる）
        q (np.ndarray): 変数

    Returns:
        np.ndarray: 導関数の値
    """
    p = 10.
    r = 28.
    b = 8. / 3.

    u = -p * q[0] + p * q[1]
    v = -q[0] * q[2]  + r * q[0] - q[1]
    w = q[0] * q[1] - b * q[2]
    
    return np.stack([u, v, w], axis=0)

In [2]:
# 初期値のlist
q0_list = [
    np.array([-30., -30., 0.]),
    np.array([30., -30., 0.]),
    np.array([-30., 30., 0.]),
    np.array([30., 30., 0.]),
    np.array([-30., -30., 60.]),
    np.array([30., -30., 60.]),
    np.array([-30., 30., 60.]),
    np.array([30., 30., 60.]),
]

# 時刻の設定
t_start = 0.
t_end = 2.
h = 0.01
t_eval = np.arange(t_start, t_end, h)

# Loretenz方程式の解を求め、listを作成
q_list = []
for q0 in q0_list:
    sol = solve_ivp(func_lorentz, [t_start, t_end], q0, t_eval=t_eval, vectorized=True)     # SciPyソルバー
    q_list.append(sol.y)    # ソルバーからLoretenz方程式の解を取得しlistに追加
q_array = np.concatenate(q_list, axis=1)

q_array.shape

(3, 1600)

In [3]:
# Traceを作成
trace = go.Scatter3d(
    x=q_array[0],       # x軸に使用する変数
    y=q_array[1],       # y軸に使用する変数
    z=q_array[2],       # z軸に使用する変数
    mode='markers',     # グラフモード（三次元散布図）
    marker={
        'size': 1,
        'opacity': 0.5,
        'color': q_array[2],
        'colorscale': 'Cividis'
    },
)   # Lorentz方程式の解の三次元散布図

# 独自テンプレートを読み込み
with open('custom_white.json') as f:
    custom_white_dict = json.load(f)
    template = Template(custom_white_dict)

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Scatter 3D sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [4]:
# 解の位置でのベクトルとその大きさを算出
dqdt_list = []
value_list = []
for q in q_list:
    dqdt = func_lorentz(None, q)
    dqdt_list.append(dqdt)
    value_list.append(np.linalg.norm(dqdt, axis=0))
dqdt_array = np.concatenate(dqdt_list, axis=1)      # コーンプロットで使用
value_array = np.concatenate(value_list, axis=0)    # 三次元バブルチャートで使用

print(dqdt_array.shape)
print(value_array.shape)

(3, 1600)
(1600,)


In [5]:
max_size = 12   # マーカーの最大サイズ
sizes = value_array / value_array.max() * max_size

# Traceを作成
traces = go.Scatter3d(
    x=q_array[0],
    y=q_array[1],
    z=q_array[2],
    mode='markers',
    marker={
        'size': sizes,
        'opacity': 0.5,
        'color': q_array[2],
        'colorscale': 'Plasma'
    },
)   # Lorentz方程式の解の三次元バブルチャート

# Figureを作成
figure = go.Figure(traces, layout)

figure

In [6]:
# Traceを作成
trace = go.Cone(
    x=q_array[0],           # x軸に使用する変数
    y=q_array[1],           # y軸に使用する変数
    z=q_array[2],           # z軸に使用する変数
    u=dqdt_array[0],        # コーンのx成分に使用する変数
    v=dqdt_array[1],        # コーンのy成分に使用する変数
    w=dqdt_array[2],        # コーンのz成分に使用する変数
    colorscale='thermal',   # カラースケール
)   # Lorentz方程式の解のコーンプロット

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Cone sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [7]:
# コーンプロットのTraceを作成してlistに追加
trace = go.Cone(
    x=q_array[0],
    y=q_array[1],
    z=q_array[2],
    u=dqdt_array[0],
    v=dqdt_array[1],
    w=dqdt_array[2],
    colorscale='haline',
)   # Lorentz方程式の解のコーンプロット
traces = [trace]

# 三次元折れ線グラフをTraceのlistに追加
for q, value in zip(q_list, value_list):
    trace = go.Scatter3d(
        x=q[0],
        y=q[1],
        z=q[2],
        mode='lines',       # グラフモード（三次元折れ線グラフ）
        line={
            'color': value,
            'colorscale': 'haline',
        },
        showlegend=False    # 凡例の表示（表示なし）
    )   # Lorentz方程式の解の三次元折れ線グラフ
    traces.append(trace)

# Figureを作成
figure = go.Figure(traces, layout)

figure

In [8]:
# x座標、y座標、z座標の格子状配列
x, y, z = np.meshgrid(
    np.linspace(-30, 30., 31, endpoint=True),
    np.linspace(-30, 30., 31, endpoint=True),
    np.linspace(0, 60., 31, endpoint=True),
)

# 格子状配列の位置におけるベクトルの大きさ
q = np.stack([x, y, z], axis=0)         # xyzをまとめた変数
dqdt = func_lorentz(None, q)            # ベクトル
value = np.linalg.norm(dqdt, axis=0)    # ベクトルの大きさ

print(x.shape)
print(value.shape)

(31, 31, 31)
(31, 31, 31)


In [9]:
# Traceを作成
trace = go.Isosurface(
    x=x.flatten(),          # x軸に使用する変数
    y=y.flatten(),          # y軸に使用する変数
    z=z.flatten(),          # z軸に使用する変数
    value=value.flatten(),  # 強度に使用する変数
    isomin=value.min(),     # 等値面の最小値
    isomax=value.max(),     # 等値面の最大値
    opacity=0.5,            # 透過率
    surface_count=11,       # 等値面の個数
    colorscale='deep',      # カラースケール
)   # ベクトル大きさの等値面図

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Isosurface sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [10]:
# Traceを作成
trace = go.Isosurface(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    value=value.flatten(),
    isomin=value.min(),
    isomax=value.max(),
    opacity=0.5,
    surface_count=11,
    colorscale='deep',
    caps={
        'x_show': False,
        'y_show': False,
        'z_show': False
    }   # 端部の表示設定（xyzすべて表示なし）
)   # ベクトル大きさの等地面図（端部表示なし）

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [11]:
# Traceを作成
trace = go.Volume(
    x=x.flatten(),          # x軸に使用する変数
    y=y.flatten(),          # y軸に使用する変数
    z=z.flatten(),          # z軸に使用する変数
    value=value.flatten(),  # 強度に使用する変数
    isomin=value.min(),     # 等値面の最小値
    isomax=value.max(),     # 等値面の最大値
    surface_count=11,       # 等値面の個数
    colorscale='deep',      # カラースケール
    opacityscale=[
        [0., 0.8],          # 正規化した強度0で透過率0.8
        [0.5, 0.2],         # 正規化した強度0.5で透過率0.2
        [1., 0.9]           # 正規化した強度1で透過率0.9
    ]
)   # ベクトル大きさのボリュームプロット

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Volume sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [12]:
# 楕円体のパラメータ
a = 4.
b = 3.
c = 2.

# x座標とy座標の格子状配列
x, y = np.meshgrid(np.linspace(-a, a, 101, endpoint=True), np.linspace(-b, b, 101, endpoint=True))

# xとyに対応するzの値
z = c * np.sqrt(1. - (x/a)**2 - (y/b)**2)

print(x.shape)
print(y.shape)
print(z.shape)

(101, 101)
(101, 101)
(101, 101)


C:\Users\eizo8\AppData\Local\Temp\ipykernel_22256\1318678907.py:10: RuntimeWarning: invalid value encountered in sqrt
  z = c * np.sqrt(1. - (x/a)**2 - (y/b)**2)


In [13]:
z

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(101, 101))

In [15]:
# Traceを作成
trace = go.Surface(
    x=x,                # x軸に使用する変数
    y=y,                # y軸に使用する変数
    z=z,                # z軸に使用する変数
    colorscale='algae'  # カラースケール
)   # 楕円体のサーフェスプロット（z>0）

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Surface sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [16]:
# 楕円体の媒介変数
theta = np.linspace(0, np.pi, 50, endpoint=True)
phi = np.linspace(0, 2.*np.pi, 100, endpoint=True)

# 媒介変数の格子状配列
theta2, phi2 = np.meshgrid(theta, phi)

# 媒介変数からx座標、y座標、z座標を算出
x = 4. * np.sin(theta2) * np.cos(phi2)
y = 3. * np.sin(theta2) * np.sin(phi2)
z = 2. * np.cos(theta2)

print(x.shape)
print(y.shape)
print(z.shape)

(100, 50)
(100, 50)
(100, 50)


In [17]:
# Traceを作成
trace = go.Scatter3d(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    mode='markers',
    marker={
        'size': 1,
        'opacity': 0.2
    }
)   # 楕円体の三次元散布図

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Scatter 3D sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [17]:
# Traceを作成
traces = go.Surface(
    x=x,
    y=y,
    z=z,
    colorscale='matter'
)   # 楕円体のサーフェスプロット（媒介変数表示）

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Surface sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(traces, layout)

figure

In [18]:
# Traceを作成
trace = go.Surface(
    z=z,
    colorscale='matter'
)   # z座標のみでのサーフェスプロット（失敗例）

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [19]:
# 単位球表面のランダムな座標を算出
q = np.random.randn(3, 1000)
q /= np.linalg.norm(q, axis=0)

# 楕円体表面のランダムな座標に変換
q[0] *= 4.
q[1] *= 3.
q[2] *= 2.

q.shape

(3, 1000)

In [20]:
# Traceを作成
trace = go.Scatter3d(
    x=q[0],
    y=q[1],
    z=q[2],
    mode='markers',
    marker={
        'size': 1,
        'opacity': 0.2
    }
)   # 楕円体の三次元散布図（ランダム位置）

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Scatter 3D sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [21]:
# Traceを作成
trace = go.Mesh3d(
    x=q[0],                 # x軸に使用する変数
    y=q[1],                 # y軸に使用する変数
    z=q[2],                 # z軸に使用する変数
    intensity=q[2],         # 色に使用する変数
    colorscale='Purpor',    # カラースケール
    alphahull=0             # 三角形分割アルゴリズム（凸包）
)   # 楕円体のメッシュプロット（凸包アルゴリズム）

# Layoutを作成
layout = go.Layout(
    template=template,
    title='Mesh 3D sample',
    margin={
        'r': 20,
        't': 30,
        'l': 20,
        'b': 30
    }
)

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [22]:
# Traceを作成
trace = go.Mesh3d(
    x=q[0],
    y=q[1],
    z=q[2],
    intensity=q[2],
    colorscale='Purpor',
    alphahull=-1,   # 三角形分割アルゴリズム（ドローネ三角形分割）
)   # 楕円体のメッシュプロット（ドローネ三角形分割アルゴリズム）

# Figureを作成
figure = go.Figure(trace, layout)

figure

In [23]:
# Traceを作成
trace = go.Mesh3d(
    x=q[0],
    y=q[1],
    z=q[2],
    intensity=q[2],
    colorscale='Purpor',
    alphahull=2,    # 三角形分割アルゴリズム（アルファシェイプ）
)   # 楕円体のメッシュプロット（アルファシェイプアルゴリズム）

figure = go.Figure(trace, layout)

figure